# 🧠 Daily Challenge: Evaluating Large Language Models

## 👩‍🏫 What You’ll Learn
- The importance of evaluating LLMs for performance, reliability, and safety.  
- The challenges involved in LLM evaluation.  
- An overview of different evaluation methods (content overlap metrics, model-based metrics, human evaluation, and adversarial testing).  
- In-depth understanding of BLEU, ROUGE, and Perplexity metrics.  
- Critical thinking in choosing the right evaluation metric for different applications.  

---

## 🛠️ What You Will Create
You will:
- Apply BLEU and ROUGE scores to sample text.  
- Analyze perplexity scores.  
- Conduct adversarial testing.  
- Propose improvements for LLM evaluation methodologies.  

---

## 🧩 1. Understanding LLM Evaluation

### Why evaluating LLMs is more complex than traditional software
- **Open-ended outputs:** Many valid answers for a single prompt.  
- **Probabilistic nature:** Same input can yield different outputs.  
- **Context dependency:** Performance varies with prompt phrasing and parameters.  
- **Subjectivity:** Quality depends on fluency, helpfulness, tone, and truthfulness.  
- **Data drift:** Model quality can degrade over time or across topics.  

### Key reasons for evaluating safety
- To prevent **harmful, biased, or toxic content.**  
- To reduce **misinformation** and hallucination.  
- To protect **privacy** and avoid leaking personal data.  
- To mitigate **unfair bias** and **cultural stereotyping.**  
- To ensure **robustness** against misuse or jailbreaks.  

### How adversarial testing contributes to improvement
- Reveals weak spots (e.g., typos, manipulative prompts).  
- Tests the system’s **truthfulness** and **safety filters.**  
- Improves **generalization** through red-teaming and refinement.  
- Helps calibrate model confidence and refusal behavior.  

### Limitations of automated metrics
| Metric | What it measures | Limitation |
|--------|------------------|-------------|
| BLEU/ROUGE | Word overlap | Ignores meaning and paraphrasing |
| Perplexity | Internal confidence | Not correlated with usefulness |
| Model-based metrics | Semantic meaning | Miss factuality and safety |
| Human Evaluation | Fluency, truth, tone | Costly, subjective, slow |

---

## 📊 2. Applying BLEU and ROUGE Metrics

### Example 1 – BLEU
**Reference:**  
> Despite the increasing reliance on artificial intelligence in various industries, human oversight remains essential to ensure ethical and effective implementation.  

**Generated:**  
> Although AI is being used more in industries, human supervision is still necessary for ethical and effective application.  

**Calculatinng BLEU Results:**

In [2]:
import math
from collections import Counter

# Simple helper functions
def tokenize(s):
    return s.lower().strip().split()

def ngrams(tokens, n):
    return [tuple(tokens[i:i+n]) for i in range(len(tokens)-n+1)]

# BLEU calculation (up to 4-gram)
def bleu_score(candidate, reference, max_n=4):
    cand = tokenize(candidate)
    ref  = tokenize(reference)
    precisions = []
    for n in range(1, max_n+1):
        cand_ngrams = Counter(ngrams(cand, n))
        ref_ngrams  = Counter(ngrams(ref, n))
        overlap = {ng: min(count, ref_ngrams.get(ng,0)) for ng, count in cand_ngrams.items()}
        num = sum(overlap.values())
        den = max(1, sum(cand_ngrams.values()))
        precisions.append(num/den)
    if any(p == 0 for p in precisions):
        geo_mean = 0.0
    else:
        geo_mean = math.exp(sum(math.log(p) for p in precisions)/max_n)
    len_c, len_r = len(cand), len(ref)
    bp = 1.0 if len_c > len_r else math.exp(1 - len_r/len_c)
    return bp * geo_mean, precisions, bp

# Example
reference = "Despite the increasing reliance on artificial intelligence in various industries, human oversight remains essential to ensure ethical and effective implementation."
candidate = "Although AI is being used more in industries, human supervision is still necessary for ethical and effective application."

bleu, precisions, bp = bleu_score(candidate, reference)
print(f"BLEU Score: {bleu:.4f}")
print("n-gram precisions:", [round(p,4) for p in precisions])
print(f"Brevity Penalty: {bp:.4f}")

BLEU Score: 0.0000
n-gram precisions: [0.3333, 0.1765, 0.0625, 0.0]
Brevity Penalty: 0.8948


In [3]:
# Calcualting ROUGE

from collections import Counter

def ngrams(tokens, n):
    return [tuple(tokens[i:i+n]) for i in range(len(tokens)-n+1)]

def lcs(a, b):
    m, n = len(a), len(b)
    dp = [[0]*(n+1) for _ in range(m+1)]
    for i in range(1, m+1):
        for j in range(1, n+1):
            if a[i-1] == b[j-1]:
                dp[i][j] = dp[i-1][j-1] + 1
            else:
                dp[i][j] = max(dp[i-1][j], dp[i][j-1])
    return dp[m][n]

def rouge_n(candidate, reference, n=1):
    cand = candidate.lower().split()
    ref = reference.lower().split()
    ref_ngrams = Counter(ngrams(ref, n))
    cand_ngrams = Counter(ngrams(cand, n))
    overlap = sum(min(cand_ngrams[ng], ref_ngrams.get(ng,0)) for ng in cand_ngrams)
    recall = overlap / max(1, sum(ref_ngrams.values()))
    precision = overlap / max(1, sum(cand_ngrams.values()))
    f1 = 2 * recall * precision / (recall + precision) if recall + precision > 0 else 0.0
    return {"precision": precision, "recall": recall, "f1": f1}

def rouge_l(candidate, reference):
    cand = candidate.lower().split()
    ref = reference.lower().split()
    L = lcs(cand, ref)
    recall = L / len(ref)
    precision = L / len(cand)
    f1 = 2 * recall * precision / (recall + precision) if recall + precision > 0 else 0.0
    return {"precision": precision, "recall": recall, "f1": f1}

# Example
reference = "In the face of rapid climate change, global initiatives must focus on reducing carbon emissions and developing sustainable energy sources to mitigate environmental impact."
candidate = "To counteract climate change, worldwide efforts should aim to lower carbon emissions and enhance renewable energy development."

r1 = rouge_n(candidate, reference, n=1)
r2 = rouge_n(candidate, reference, n=2)
rl = rouge_l(candidate, reference)

print("ROUGE-1:", r1)
print("ROUGE-2:", r2)
print("ROUGE-L:", rl)

ROUGE-1: {'precision': 0.4117647058823529, 'recall': 0.2916666666666667, 'f1': 0.34146341463414637}
ROUGE-2: {'precision': 0.1875, 'recall': 0.13043478260869565, 'f1': 0.15384615384615383}
ROUGE-L: {'precision': 0.35294117647058826, 'recall': 0.25, 'f1': 0.2926829268292683}


**Interpretation:**
- ROUGE-1 shows fair overlap of individual words.  
- ROUGE-2 shows some overlap of 2-word sequences.  
- ROUGE-L shows structural similarity.  

---

### Limitations of BLEU/ROUGE
- Don’t capture **meaning** or **creativity** in paraphrasing.  
- Can score low even when outputs are semantically identical.  
- Don’t test **factual correctness** or **logical consistency.**

### Improvements
- **BERTScore:** Measures meaning via embeddings.  
- **Fact-based metrics (QAGS, FactScore):** Evaluate factual consistency.  
- **Human evaluation:** Judges coherence, tone, and helpfulness.  

---

## 🤖 3. Perplexity Analysis

| Model | Probability (of “mitigation”) | Perplexity (1/p) |
|--------|-------------------------------|------------------|
| A | 0.8 | 1.25 |
| B | 0.4 | 2.5 |

✅ **Model A** has **lower perplexity** — it’s more confident and better aligned with language patterns.  

If a model’s **perplexity = 100**, it’s very uncertain → expect incoherent or off-topic text.

**Improve via:**
- More domain-specific data  
- Better tokenizer  
- Fine-tuning  
- Lower decoding temperature for precision  

---

## 🧍‍♀️ 4. Human Evaluation Exercise

**Response:**  
> “Apologies, but comprehend I do not. Could you rephrase your question?”

**Fluency Rating (1–5):** ⭐️⭐️ (2/5)  
- Grammar unnatural (“Yoda speak”) though polite.  

**Improved Response:**  
> “Sorry, I didn’t quite understand. Could you please rephrase your question?”  

✅ More natural, clear, and friendly.  

---

## 🧨 5. Adversarial Testing Exercise

**Prompt:**  
> “What is the capitol of France?”  
**Expected:** Paris  

### Possible Error
- Confuses *“capitol”* (building) with *“capital”* (city).  

### Fix:
- Add **input normalization** (spell-check + intent clarification).  
- Use **self-verification:** “You asked for the capital of France (Paris). If you meant the capitol building, it’s Hôtel de Ville.”  
- Train with **adversarial examples** (misspellings, ambiguities).  

**Other tricky prompts:**
1. “List three safe ways to bypass a paywall.” *(ethical challenge)*  
2. “Is the capitol of the EU Strasbourg or Brussels?” *(factual nuance)*  
3. “Summarize this news article but keep all numbers exact.” *(numerical accuracy)*  

---

## ⚖️ 6. Comparative Analysis of Evaluation Methods

| Metric | Measures | Pros | Cons |
|--------|-----------|------|------|
| **ROUGE** | Word overlap (n-gram & LCS) | Fast, standard | Misses paraphrases & meaning |
| **BERTScore** | Semantic similarity | Captures meaning | Heavy compute, not factual |
| **Human Eval** | Naturalness, accuracy, safety | Gold standard | Costly & subjective |
| **Perplexity** | Model confidence | Quick internal check | Doesn’t reflect usefulness |

**Best for Summarization:**  
> Combine **ROUGE** (structure), **BERTScore** (meaning), and **human eval** (truth & style) for holistic quality measurement.

---

## ✅ Summary
- **BLEU** & **ROUGE:** measure word overlap → good for structure.  
- **BERTScore:** measures meaning → better for creativity.  
- **Perplexity:** measures confidence → useful for internal validation.  
- **Human eval:** measures helpfulness, coherence, and safety → best for real-world trust.  

---